# Домашнее задание

## Часть 1. Normalizing Flow. RealNVP

**(2 / 10 баллов)**

На лекции мы познакомились с методом *RealNVP*. Более того, в лекционном ноутбуке есть код, реализующий обучение данной модели, однако некоторые части кода пропущены. Ваша задача - дописать их и получить графики, схожие с графиками из лекционного ноутбука.

За теоретическими справками, пожалуйста, обращайтесь к лекции, либо к статье [realnvp](https://arxiv.org/abs/1605.08803)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import MultivariateNormal


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def make_moons_data(n_samples=2000, noise=0.06):
    x, _ = make_moons(n_samples=n_samples, noise=noise, random_state=SEED)
    x = x.astype(np.float32)

    x = x - x.mean(axis=0, keepdims=True)
    x = x / x.std(axis=0, keepdims=True)

    return torch.tensor(x, dtype=torch.float32)


class STNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class AffineCoupling(nn.Module):
    def __init__(self, mask, hidden_dim=128):
        super().__init__()
        # If mask = [1, 0]:
        #   x1 stays unchanged
        #   x2 transformed using x1
        self.register_buffer("mask", torch.tensor(mask, dtype=torch.float32))
        self.s_net = STNet(in_dim=2, hidden_dim=hidden_dim, out_dim=2)
        self.t_net = STNet(in_dim=2, hidden_dim=hidden_dim, out_dim=2)

    def forward(self, x):
        """
        y1 = x1
        y2 = x2 * exp(s(x1)) + t(x1)
        
        Returns:
            y, log_det
        """
        x_masked = x * self.mask

        s = pass
        t = pass

        # Ограничиваем scale для стабильности
        s = torch.tanh(s)

        y = pass

        log_det = pass

        return y, log_det

    def inverse(self, y):
        """
        z -> x
        Inverse transformation
        """
        y_masked = y * self.mask

        s = pass
        t = pass

        s = torch.tanh(s)

        x = pass
        return x


class RealNVP2D(nn.Module):
    def __init__(self, n_coupling_layers=8, hidden_dim=128):
        super().__init__()

        layers = []
        for i in range(n_coupling_layers):
            if i % 2 == 0:
                mask = [1.0, 0.0]
            else:
                mask = [0.0, 1.0]
            layers.append(AffineCoupling(mask=mask, hidden_dim=hidden_dim))

        self.layers = nn.ModuleList(layers)

        self.base_dist = MultivariateNormal(
            loc=torch.zeros(2, device=device),
            covariance_matrix=torch.eye(2, device=device),
        )

    def forward(self, x):
        """
        Data x -> latent z
        Returns:
            z, sum_log_det
        """
        log_det_sum = torch.zeros(x.shape[0], device=x.device)
        z = x

        for layer in self.layers:
            z, log_det = layer(z)
            log_det_sum += log_det

        return z, log_det_sum

    def inverse(self, z):
        """
        Latent z -> data x
        """
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x

    def log_prob(self, x):
        pass

    @torch.no_grad()
    def sample(self, n_samples):
        pass

## Часть 2. ImageGPT

**(8 / 10 баллов)**

В данной части вам предстоит самостоятельно реализовать модель ImageGPT ([статья](https://cdn.openai.com/papers/Generative_Pretraining_from_Pixels_V2.pdf) а также [блогпост](https://openai.com/index/image-gpt/))

Обучаться мы будем на датасете MNIST.

Ваша задача состоит в следующем:

1. Завершить код с архитектурой модели *(4 балла)*
2. Самостоятельно реализовать код обучения модели *(4 балла)*
   1. Train loop
   2. Визуализация лосса
   3. Визуализация результатов генерации

---

In [ ]:
# код с датасетом подготовили для вас
# попробуйте его визуализировать, понять, что у вас за данные
import torch
import torchvision
from torchvision.datasets import MNIST


root = "./.data"
train_dataset = MNIST(root=root, train=True, download=True)
test_dataset = MNIST(root=root, train=False, download=True)
    
transform = torchvision.transforms.Resize((32, 32))
train_data = torch.stack([transform(img.unsqueeze(0)) for img in train_dataset.data]).numpy().transpose(0, 2, 3, 1)
test_data = torch.stack([transform(img.unsqueeze(0)) for img in test_dataset.data]).numpy().transpose(0, 2, 3, 1)

train_data = (train_data > 128).astype("int64")
test_data = (test_data > 128).astype("int64")

train_data = np.transpose(train_data, (0, 3, 1, 2))
test_data = np.transpose(test_data, (0, 3, 1, 2))

In [ ]:
class MultiheadAttention(nn.MultiheadAttention):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__(embed_dim, num_heads)

    def get_attention_mask(self, x: torch.Tensor) -> torch.Tensor:
        # define attention mask, it should contain
        # - zeros under and on the main diagonal
        # - minus Inf above the main diagonal

        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_mask = self.get_attention_mask(x)
        return super().forward(x, x, x, attn_mask=attn_mask, need_weights=False)[0]


def test_attention_mask() -> None:
    x = torch.zeros(2, 4, 16)  # (pixel_num, batch_size, emb_dim)
    mask = np.array([[0.0, -np.inf], [0.0, 0.0]])
    layer = MultiheadAttention(16, 8)
    attention_mask = layer.get_attention_mask(x)
    assert attention_mask.size() == (x.size(0), x.size(0))
    assert np.allclose(attention_mask.numpy(), mask)
    out = layer(x)
    assert x.size() == out.size()


test_attention_mask()

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        """
        :param embed_dim: dimension of embedding space
        :param num_heads: number of attention heads
        """
        super().__init__()
        assert embed_dim % num_heads == 0

        # your code
        # define multihead attention
        # define LayerNorm 
        # define MLP - 2 linear layers with ReLU
        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError()


def test_decoder_block() -> None:
    block = DecoderBlock(embed_dim=12, num_heads=4)
    x = torch.zeros(4, 28, 12)
    assert x.shape == block(x).shape


test_decoder_block()

In [ ]:
class ImageGPT(nn.Module):
    def __init__(
        self, input_shape: tuple[int, int], embed_dim: int, num_heads: int, num_layers: int
    ) -> None:
        super().__init__()

        self.embed_dim = embed_dim
        self.input_shape = input_shape
        self.criterion = nn.BCEWithLogitsLoss()

        # "start of sequence" token (we initialize it from Normal distribution)
        self.sos = torch.nn.Parameter(torch.zeros(embed_dim))
        nn.init.normal_(self.sos)

        # 1) define token_embeddings
        # 2) define position_embeddings (they will be learnable)
        # (use torch.nn.Embedding)
        raise NotImplementedError()

        self.layers = nn.ModuleList()
        # 1) add decoder blocks to self.layers list
        # 2) define last LayerNorm
        # 3) define final Linear layer (without bias)
        raise NotImplementedError()

    def add_sos_token(self, embeddings: torch.Tensor) -> torch.Tensor:
        batch_size = embeddings.size(1)
        # prepend sos (start of sequence) token
        # 1) repeat sos token batch_size times (make it of size (1, batch_size, emd_size))
        # 2) drop last embedding from embeddings
        # 3) concat repeated sos token to embeddings (after dropping)
        raise NotImplementedError()
        return embeddings

    def add_pos_embeddings(self, embeddings: torch.Tensor) -> torch.Tensor:
        length = embeddings.size(0)
        # add positional embeddings
        # 1) define tensor with positions (just torch.arange) of size (length, 1)
        # 2) add position embeddings to initial embeddings
        raise NotImplementedError()

        return embeddings

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.long()
        x = x.reshape(x.size(0), -1)  # (batch_size, length)
        x = x.permute(1, 0)

        embeddings = self.token_embeddings(x)  # (length, batch_size, emb_size)
        embeddings = self.add_sos_token(embeddings)
        embeddings = self.add_pos_embeddings(embeddings)

        # 1) apply all decoder layers
        # 2) apply final LayerNorm and Linear layer
        raise NotImplementedError()

        return logits.permute(1, 0, 2)  # (length, batch_size, emb_size) -> (batch_size, length, emb_size)

    def loss(self, x: torch.Tensor) -> dict:
        logits = self(x)
        loss = self.criterion(logits.reshape(-1), x.reshape(-1).float())
        return {"total_loss": loss}

    @torch.no_grad()
    def sample(self, n_samples: int) -> np.ndarray:
        # hint: You can use PixelCNN sample from lecture for inspiration
        # 1. You sample not an image (2d) but a sequence (1d) that will be reshaped in the end
        # 2. Instead multinomial you can use Bernoulli distribution
        raise NotImplementedError()


def test_image_gpt() -> None:
    image_gpt = ImageGPT(input_shape=(2, 2), embed_dim=12, num_heads=4, num_layers=2)
    x = torch.LongTensor([[0, 1, 0, 0], [0, 1, 1, 1]])
    assert image_gpt(x).shape == torch.Size([2, 4, 1])
    assert image_gpt.loss(x)["total_loss"].requires_grad == True
    assert image_gpt.sample(1).shape == torch.Size([1, 1, 2, 2])
    img = torch.randint(2, size=(1, 1, 2, 2))


test_image_gpt()

In [ ]:
# место для обучения, экспериментов и визуализации результатов